In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGSMITH_TRACING"] = "false"

## workflow 
```
General Document → Summary → Generate Questions → Extract Answers → Generate Q&A pairs → Save as CSV file
```

In [2]:
from pydantic import BaseModel, Field
from typing import List


class GroundTruth(BaseModel):
    question: str = Field(..., description = "The question asked by the user.")
    answer: str = Field(description = "The correct answer to the user's question.")

class GroundTruthList(BaseModel):
    ground_truths: List[GroundTruth] = Field(description = "A list of ground truth question-answer pairs.")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.0,)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model = model,
    tools = [], 
    response_format = GroundTruthList,
    system_prompt = """You are an expert at generating ground truth datasets for evaluating AI agents. 
You will be given a document from which you will generate the ground truth data.
Follow these steps below to perform the task 

1. Read the provided document carefully.
2. Think about the purpose of the document and the type of information it contains.
3. Generate a list of question-aswer pairs that accurately reflect the content of the document.
4. If you want to mention the document, make sure to include the title""",
)

# Load Documents

In [5]:
from pathlib import Path

markdown_path = Path("documents_with_english_titles_markdown/employee_benefits_and_welfare_guide.md")
markdown_content = markdown_path.read_text(encoding="utf-8")

markdown_content

'직원 복리후생 및 복지 가이드\n=========================\n\n제 1 장 복리후생 개요\n---------------------\n\n제 1 조 (목적)\n\n* 본 가이드는 직원들에게 제공되는 복리후생 제도를 명확히 안내하여, 회사 생활의 만족도를 높이고, 근로 의욕을 증진하기 위함이다.\n\n제 2 조 (적용 대상)\n\n* 정규직, 계약직, 인턴 등 모든 직원에게 적용된다.\n\n제 2 장 건강 관리 및 보험\n-------------------------\n\n제 3 조 (건강 보험)\n\n* 전 직원은 국민건강보험에 가입되며, 회사가 일정 비율의 보험료를 부담한다.\n* 추가로 회사는 단체 실손 보험을 제공하여, 입원 및 통원 치료비를 지원한다.\n\n제 4 조 (정기 건강 검진)\n\n* 매년 1회 정기 건강 검진을 제공한다.\n* 검진 항목: 기본 검진 (혈액 검사, X-ray) + 종합 검진 옵션\n* 검진 비용: 회사 100% 부담\n\n제 3 장 휴가 및 유연 근무\n-------------------------\n\n제 5 조 (연차 유급 휴가)\n\n* 근속 연수에 따른 연차 유급 휴가 지급\n\n| 근속 연수 | 연차 휴가 일수 |\n|-----------|----------------|\n| 1년 미만    | 월 1일          |\n| 1~3년      | 15일           |\n| 4~6년      | 17일           |\n| 7~9년      | 19일           |\n| 10년 이상   | 20일           |\n\n- 연차 휴가는 본인 승인으로 자유롭게 사용할 수 있다.\n\n## 제 6 조 (출산 및 육아 휴가)\n\n- 출산 휴가: 20일 (유급)\n- 육아 휴가: 최대 1년 (첫 3개월 50% 급여 지원)\n\n## 제 7 조 (경조 휴가)\n\n- 다음의 경조사 사유에 대해 휴가를 부여한다:\n\n| 경조 사유     | 휴가 일수 |\n|-----------

In [ ]:
# Agent에 문서를 전달하여 Q&A 생성 실행
benefits_dataset = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"""Generate a ground truth dataset based on the following document:
{markdown_content}
"""
    }]
})

### Tool

In [6]:
from langchain.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

@tool
def generate_QnA_list(document: str, summary: str) -> GroundTruthList:
    """
    Generate a list of question-answer pairs based on the provided document and summary.
    
    Args:
        document (str): The content of the document to generate Q&A from.
        summary (str): A brief summary of the document's content.
        
    Returns:
        GroundTruthList: A list of ground truth question-answer pairs.
    """

    system_prompt = """You are a helpful assistant that generates a list of questions and answers from the given markdown file. 
    Follow the steps below in order to generate the list of questions and answers:
    1. Look at the summary the content of the markdown file to understand the purpose of the document.
    2. Generate a list of questions and answers based on the summarized content.
    3. Make sure to format the question and answer in Korean as the original question and answer is in Korean.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Text: {document}\nSummary: {summary}"),
    ])
    
    model_with_json = model.with_structured_output(GroundTruthList)
    chain = prompt | model_with_json
    
    return chain.invoke({"document": document, "summary": summary})


@tool
def summarize_document(text: str) -> str:
    """
    Summarize the provided document.
    
    Args:
        document (str): The content of the document to summarize.
        
    Returns:
        str: A brief summary of the document's content."""


    summary_chain = model | StrOutputParser()
    return summary_chain.invoke(f"Summarize the following document:\n\n{text}")


In [7]:
agent_with_tools = create_agent(
    model = model,
    tools = [generate_QnA_list, summarize_document], 
    response_format = GroundTruthList,
    system_prompt = """You are an expert at generating ground truth datasets for evaluating AI agents. 
You will be given a document from which you will generate the ground truth data.
You have access to two tools: analyze_document and generate_qna_list.
The analyze_document tool helps you summarize the document, while the generate_qna_list tool generates the Q&A pairs based on the document and its summary.
Use these tools effectively to create a comprehensive ground truth dataset.""",
)

In [ ]:
qna_dataset = agent_with_tools.invoke({
    "messages": [{
        "role": "user",
        "content": f"""Generate a ground truth dataset based on the following document:
    {markdown_content}
    """
    }]
})

In [ ]:
# save as .csv file
import pandas as pd

# structured_response에서 Q&A 리스트 추출
df = pd.DataFrame([gt.model_dump() for gt in qna_dataset['structured_response'].ground_truths])

# CSV로 저장
df.to_csv("./welfare_list_with_agent_tools.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} Q&A pairs to output/welfare_list_with_agent_tools.csv")
df